In [0]:
from pyspark.sql.functions import col, to_date, date_format, when, lit, regexp_replace
import requests
import pandas as pd
import io

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
from bs4 import BeautifulSoup

page_url = "https://revenue.scot/news-publications/publications/statistics/land-buildings-transaction-tax-statistics/monthly-land-buildings-transaction-tax-lbtt"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(page_url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
ods_links = [a["href"] for a in soup.find_all("a", href=True) if ".ods" in a["href"]]
print(ods_links)

In [0]:
base_url = "https://revenue.scot"
ods_url = base_url + ods_links[0]
print(f"Downloading: {ods_url}")

ods_response = requests.get(ods_url, headers=headers)
ods_response.raise_for_status()

import subprocess
subprocess.run(["pip", "install", "odfpy", "--quiet"])

ods_bytes = io.BytesIO(ods_response.content)
xl = pd.ExcelFile(ods_bytes, engine="odf")
print(xl.sheet_names)

In [0]:
ods_bytes = io.BytesIO(ods_response.content)
df_t1_raw = pd.read_excel(ods_bytes, sheet_name="Table_1", engine="odf", header=4)
print(df_t1_raw.shape)
print(df_t1_raw.columns.to_list())
print(df_t1_raw.head(10))

In [0]:
ods_bytes = io.BytesIO(ods_response.content)
df_t1 = pd.read_excel(ods_bytes, sheet_name="Table_1", engine="odf", header=4)

df_t1 =df_t1.dropna(subset=["Month"])

df_t1.columns = [
    "month",
    "residential_returns",
    "residential_tax_ex_ads",
    "residential_ads",
    "residential_tax_total",
    "non_residential_returns",
    "non_residential_tax_ex_ads",
    "non_residential_ads",
    "non_residential_tax_total",
    "total_returns",
    "total_tax_ex_ads",
    "total_ads",
    "total_tax"
]

df_bronze_t1 = spark.createDataFrame(df_t1)
(
    df_bronze_t1.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .saveAsTable("bronze.lbtt_table_1")
)

spark.sql("SELECT * FROM bronze.lbtt_table_1 LIMIT 5").show()
spark.sql("SELECT COUNT(*) FROM bronze.lbtt_table_1").show()

In [0]:
ods_bytes = io.BytesIO(ods_response.content)
df_t2 = pd.read_excel(ods_bytes, sheet_name="Table_2", engine="odf", header=4)
print(df_t2.shape)
print(df_t2.columns.tolist())

In [0]:
ods_bytes = io.BytesIO(ods_response.content)
df_t2 = pd.read_excel(ods_bytes, sheet_name="Table_2", engine="odf", header=4)
df_t2 = df_t2.dropna(subset=["Month"])

df_t2.columns = [
    "month",
    "band_up_to_145k_returns",
    "band_up_to_145k_tax",
    "band_145k_to_250k_returns",
    "band_145k_to_250k_tax",
    "band_250k_to_325k_returns",
    "band_250k_to_325k_tax",
    "band_325k_to_750k_returns",
    "band_325k_to_750k_tax",
    "band_over_750k_returns",
    "band_over_750k_tax",
    "total_returns",
    "total_tax_ex_ads"
]

df_bronze_t2 = spark.createDataFrame(df_t2)

(
    df_bronze_t2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.lbtt_table_2")
)

spark.sql("SELECT * FROM bronze.lbtt_table_2 LIMIT 5").show()
spark.sql("SELECT COUNT(*) FROM bronze.lbtt_table_2").show()